<a href="https://colab.research.google.com/github/Sathvik1500/generative-ai-for-beginners/blob/main/Hackathon_2026_Submission_2_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from google.colab import drive
from google.colab import files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
travel_train = pd.read_csv('/content/drive/MyDrive/MIT/Traveldata_train_(1)_(1).csv')
survey_train = pd.read_csv('/content/drive/MyDrive/MIT/Surveydata_train_(1)_(1).csv')

travel_test = pd.read_csv('/content/drive/MyDrive/MIT/Traveldata_test_(1)_(1).csv')
survey_test = pd.read_csv('/content/drive/MyDrive/MIT/Surveydata_test_(1)_(1).csv')

In [ ]:
# Ensure ID consistency
for df in [travel_train, survey_train, travel_test, survey_test]:
    df["ID"] = df["ID"].astype(str)

# Merge
train = travel_train.merge(survey_train, on="ID", how="inner")
test = travel_test.merge(survey_test, on="ID", how="inner")

print(train.shape, test.shape)

(94379, 25) (35602, 24)


In [ ]:
for df in [train, test]:

    df["Total_Delay"] = (
        df["Departure_Delay_in_Mins"].fillna(0)
        + df["Arrival_Delay_in_Mins"].fillna(0)
    )

    df["Delay_Difference"] = (
        df["Arrival_Delay_in_Mins"].fillna(0)
        - df["Departure_Delay_in_Mins"].fillna(0)
    )

    df["Has_Delay"] = (
        (df["Departure_Delay_in_Mins"].fillna(0) > 0) |
        (df["Arrival_Delay_in_Mins"].fillna(0) > 0)
    ).astype(int)

    df["Delay_Ratio"] = (
        (df["Arrival_Delay_in_Mins"].fillna(0) + 1) /
        (df["Departure_Delay_in_Mins"].fillna(0) + 1)
    )

    df["Long_Distance"] = (
        df["Travel_Distance"] > df["Travel_Distance"].median()
    ).astype(int)

In [ ]:
y = train["Overall_Experience"]

X = train.drop(["ID", "Overall_Experience"], axis=1)
X_test = test.drop(["ID"], axis=1)

In [ ]:
for col in X.columns:

    if X[col].dtype == "object":
        X[col] = X[col].fillna("Missing")
        X_test[col] = X_test[col].fillna("Missing")

    else:
        median = X[col].median()
        X[col] = X[col].fillna(median)
        X_test[col] = X_test[col].fillna(median)

In [ ]:
cat_features = np.where(X.dtypes == "object")[0]

In [ ]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

test_preds = np.zeros(len(X_test))

fold_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):

    print(f"\n===== FOLD {fold+1} =====")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=4000,
        depth=10,
        learning_rate=0.03,
        loss_function="Logloss",
        eval_metric="Accuracy",
        random_seed=42,
        verbose=200
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_val, y_val),
        early_stopping_rounds=300,
        use_best_model=True
    )

    val_preds = model.predict(X_val)
    acc = accuracy_score(y_val, val_preds)

    print("Fold Accuracy:", acc)
    fold_scores.append(acc)

    # accumulate probabilities
    test_preds += model.predict_proba(X_test)[:, 1] / 5

print("\nMean CV Accuracy:", np.mean(fold_scores))


===== FOLD 1 =====
0:	learn: 0.8477941	test: 0.8442996	best: 0.8442996 (0)	total: 596ms	remaining: 39m 44s
200:	learn: 0.9657762	test: 0.9535919	best: 0.9535919 (199)	total: 2m 54s	remaining: 54m 51s
400:	learn: 0.9743719	test: 0.9563997	best: 0.9563997 (400)	total: 5m 42s	remaining: 51m 12s
600:	learn: 0.9809412	test: 0.9567175	best: 0.9570354 (588)	total: 8m 35s	remaining: 48m 37s
800:	learn: 0.9860800	test: 0.9574592	best: 0.9575652 (786)	total: 11m 34s	remaining: 46m 14s
1000:	learn: 0.9903050	test: 0.9577241	best: 0.9577241 (1000)	total: 14m 32s	remaining: 43m 34s
1200:	learn: 0.9931129	test: 0.9572473	best: 0.9579890 (1091)	total: 17m 24s	remaining: 40m 34s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 0.9579889807
bestIteration = 1091

Shrink model to first 1092 iterations.
Fold Accuracy: 0.9579889807162535

===== FOLD 2 =====
0:	learn: 0.8808657	test: 0.8821784	best: 0.8821784 (0)	total: 1.05s	remaining: 1h 10m
200:	learn: 0.9655643	test: 0.9564526	best: 0

In [ ]:
final_preds = (test_preds > 0.5).astype(int)

submission = pd.DataFrame({
    "ID": test["ID"],
    "Overall_Experience": final_preds
})

submission.to_csv("submission.csv", index=False)

files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>